# AeroNetra — Video Tracking & Unique Vehicle Counts (Phase 3)

**Purpose:** drive a detector frame by frame through a tracker so each vehicle keeps one identity, then report the *unique* vehicle count rather than a per-image box count.

**What you need to attach**
1. **A clip** — any `.mp4` / `.avi` / `.mov` uploaded as a Kaggle dataset. A dataset named `aeronetra-video-clip` is preferred but not required; the cell below finds the first video anywhere under `/kaggle/input`.
2. **Weights** — optional. If a project-trained `.pt` is attached (e.g. `aeronetra-trained-weights` from notebook 02), it is used with the VisDrone class list. If none is found, the notebook falls back to COCO `yolo11n.pt`, **says so loudly**, and counts only COCO vehicle classes (car, motorcycle, bus, truck).

**Tracker:** ByteTrack by default. Switch to `botsort.yaml` for better accuracy on UAV footage (it compensates for camera motion) at roughly four times the runtime.

Self-contained: the tracking helpers below are inlined from `src/aeronetra/tracking/ops.py`, so no `pip install aeronetra` is needed.

In [ ]:
# ============================================================
# Cell 1: Setup
# ============================================================
!pip install -q ultralytics

import csv
import json
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

In [ ]:
# ============================================================
# Cell 2: Data structures (inlined from aeronetra.detection.types)
# ============================================================

@dataclass
class BoundingBox:
    xmin: float
    ymin: float
    xmax: float
    ymax: float

    @property
    def xyxy(self):
        return (self.xmin, self.ymin, self.xmax, self.ymax)


@dataclass
class Detection:
    box: BoundingBox
    class_id: int
    class_name: str
    confidence: float


@dataclass
class TrackedDetection:
    """A detection plus the identity the tracker assigned to it."""
    detection: Detection
    track_id: int
    frame_index: int = 0

In [ ]:
# ============================================================
# Cell 3: Tracking helpers (inlined from aeronetra.tracking.ops)
# ============================================================

def track_class(detections):
    """The class a whole track is reported as: its most frequent class id."""
    counts = Counter(d.class_id for d in detections)
    return min(counts, key=lambda cid: (-counts[cid], cid))


def summarize(frames, min_len=1, names=None):
    """Counts unique vehicles; min_len drops tracks too short to be traffic."""
    names = names or {}
    per_track = defaultdict(list)
    for frame in frames:
        for det in frame:
            per_track[det.track_id].append(det)
    kept = [dets for dets in per_track.values() if len(dets) >= min_len]
    per_class = Counter()
    for dets in kept:
        cid = track_class(dets)
        per_class[names.get(cid, str(cid))] += 1
    return {
        "frames": len(frames),
        "unique": len(kept),
        "raw_ids": len(per_track),
        "dropped": len(per_track) - len(kept),
        "per_class": dict(sorted(per_class.items())),
    }


def color_for(track_id):
    """A stable colour per track id, so the same vehicle keeps one colour."""
    hue = int((track_id * 137) % 180)
    blue, green, red = cv2.cvtColor(
        np.uint8([[[hue, 200, 255]]]), cv2.COLOR_HSV2BGR
    )[0, 0]
    return int(blue), int(green), int(red)


def draw_tracks(image, tracked):
    for item in tracked:
        x1, y1, x2, y2 = (int(round(v)) for v in item.detection.box.xyxy)
        color = color_for(item.track_id)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {item.track_id}", (x1, max(y1 - 6, 12)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA,
        )
    return image

In [ ]:
# ============================================================
# Cell 4: Find the clip and the weights
# ============================================================
INPUT_ROOT = Path("/kaggle/input")
CLIP_SLUG = "aeronetra-video-clip"
WEIGHTS_SLUG = "aeronetra-trained-weights"
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

# COCO vehicle classes, used only when no project weights are attached.
COCO_VEHICLE_CLASSES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
VISDRONE_CLASSES = {
    0: "car", 1: "van", 2: "truck", 3: "tricycle",
    4: "awning-tricycle", 5: "bus", 6: "motor", 7: "bicycle",
}


def _videos(root):
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.suffix.lower() in VIDEO_EXTS)


# Prefer the named slug, but fall back to any video so a differently named
# dataset still works instead of failing on a technicality.
clip_candidates = _videos(INPUT_ROOT / CLIP_SLUG) or _videos(INPUT_ROOT)
assert clip_candidates, (
    f"No video found under {INPUT_ROOT}. Attach a clip as a Kaggle dataset "
    f"(ideally named '{CLIP_SLUG}')."
)
VIDEO_PATH = clip_candidates[0]

weights_root = INPUT_ROOT / WEIGHTS_SLUG
weights_candidates = (
    sorted(weights_root.rglob("*.pt")) if weights_root.exists() else []
)

if weights_candidates:
    WEIGHTS_PATH = str(weights_candidates[0])
    CLASS_NAMES = VISDRONE_CLASSES
    print(f"Using project weights: {WEIGHTS_PATH}")
    print("Class list: VisDrone (car, van, truck, tricycle, awning-tricycle, bus, motor, bicycle)")
else:
    # Explicit, not silent: this downloads COCO weights and changes what the
    # counts mean, so the fallback announces itself rather than surprising you.
    WEIGHTS_PATH = "yolo11n.pt"
    CLASS_NAMES = COCO_VEHICLE_CLASSES
    print("!" * 60)
    print("No project-trained weights found under", weights_root)
    print("Falling back to COCO 'yolo11n.pt' (Ultralytics downloads it).")
    print("Counting only COCO vehicle classes: car, motorcycle, bus, truck.")
    print("A VisDrone-trained model will be much better on aerial footage.")
    print("!" * 60)

TRACKER = "bytetrack.yaml"   # or "botsort.yaml" (more accurate, ~4x slower)
CONF, IOU = 0.25, 0.45
MIN_TRACK_LENGTH = 5         # frames a track must appear in to count
MAX_FRAMES = 900             # keeps a long clip inside a Kaggle session; None = all

print("\nclip    :", VIDEO_PATH)
print("weights :", WEIGHTS_PATH)
print("tracker :", TRACKER)
print("device  :", DEVICE)

In [ ]:
# ============================================================
# Cell 5: Track the clip, write an annotated video and save the numbers
# ============================================================
from ultralytics import YOLO

model = YOLO(WEIGHTS_PATH)

capture = cv2.VideoCapture(str(VIDEO_PATH))
assert capture.isOpened(), f"Could not open {VIDEO_PATH}"

fps = capture.get(cv2.CAP_PROP_FPS) or 25.0
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
out_path = WORK_DIR / "tracked.mp4"
writer = cv2.VideoWriter(
    str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
)
assert writer.isOpened(), "Could not open the video writer"

frames = []
rows = []
seen_ids = set()
preview = []
index = 0
try:
    while MAX_FRAMES is None or index < MAX_FRAMES:
        ok, frame = capture.read()
        if not ok:
            break

        # persist=True is what makes an identity survive between frames.
        result = model.track(
            frame, persist=True, tracker=TRACKER, conf=CONF, iou=IOU,
            device=DEVICE, verbose=False,
        )[0]

        tracked = []
        if result.boxes is not None and result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy()
            track_ids = result.boxes.id.cpu().numpy()
            for box, conf, cls_id, track_id in zip(boxes, confs, classes, track_ids):
                cid = int(cls_id)
                if cid not in CLASS_NAMES:
                    continue  # keep the count about vehicles, not every COCO class
                tracked.append(TrackedDetection(
                    detection=Detection(
                        box=BoundingBox(*(float(v) for v in box)),
                        class_id=cid,
                        class_name=CLASS_NAMES[cid],
                        confidence=float(conf),
                    ),
                    track_id=int(track_id),
                    frame_index=index,
                ))
                rows.append({
                    "frame": index,
                    "track_id": int(track_id),
                    "class_id": cid,
                    "class_name": CLASS_NAMES[cid],
                    "confidence": round(float(conf), 4),
                    "x1": round(float(box[0]), 1),
                    "y1": round(float(box[1]), 1),
                    "x2": round(float(box[2]), 1),
                    "y2": round(float(box[3]), 1),
                })

        for item in tracked:
            seen_ids.add(item.track_id)
        frames.append(tracked)

        annotated = draw_tracks(frame.copy(), tracked)
        cv2.putText(
            annotated, f"unique: {len(seen_ids)}", (12, 28),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2, cv2.LINE_AA,
        )
        writer.write(annotated)
        if index in (0, 30, 60) and len(preview) < 3:
            preview.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        index += 1
finally:
    capture.release()
    writer.release()

summary = summarize(frames, min_len=MIN_TRACK_LENGTH, names=CLASS_NAMES)
summary["weights"] = WEIGHTS_PATH
summary["tracker"] = TRACKER
summary["confidence_threshold"] = CONF
summary["iou_threshold"] = IOU

(WORK_DIR / "tracking_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
if rows:
    with open(WORK_DIR / "tracks.csv", "w", newline="", encoding="utf-8") as handle:
        writer_csv = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer_csv.writeheader()
        writer_csv.writerows(rows)

print(json.dumps(summary, indent=2))
print("\nannotated video :", out_path)
print("summary json    :", WORK_DIR / "tracking_summary.json")
print("per-frame rows  :", len(rows))

In [ ]:
# ============================================================
# Cell 6: Look at the result (three annotated frames)
# ============================================================
if preview:
    figure, axes = plt.subplots(1, len(preview), figsize=(6 * len(preview), 4))
    axes = np.atleast_1d(axes)
    for axis, image in zip(axes, preview):
        axis.imshow(image)
        axis.axis("off")
    figure.suptitle(f"{TRACKER} · unique vehicles: {summary['unique']}")
    plt.tight_layout()
    plt.show()
else:
    print("No frames were processed — check that the clip is readable.")

## Reading the result

`unique` is the number of distinct identities that survived the minimum-length filter — that is the vehicle count. `raw_ids` is how many identities the tracker created in total, so `dropped` is the flicker: short-lived ids that were almost certainly false positives rather than vehicles.

If `dropped` is a large share of `raw_ids`, the detector is noisy at this confidence and raising `CONF` (or `MIN_TRACK_LENGTH`) will clean the count up. If two boxes on one vehicle carry different ids, the tracker is fragmenting a track — that is the case `track_buffer` and `botsort.yaml` exist to address.

The annotated video is the check: the count in the corner should only rise when a genuinely new vehicle appears, never while one is merely occluded. `tracks.csv` lets you audit any single identity frame by frame.

**Honest caveat:** this notebook reports what the tracker produced on your clip. It is not a benchmark, and a unique count from one video is not evidence about detection accuracy.